In [1]:

import sys
import logging
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL

import pandas as pd
import numpy as np
from QGI.neoapi import Neo4j
from QGI.feishu import *
import os

In [2]:
def get_logger(name,level=None,file=None):
    logger=logging.getLogger(name)
    logger.setLevel(logging.INFO)
    
    ch=logging.StreamHandler()
    ch.setLevel(logging.INFO)
    formatter=logging.Formatter(fmt='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',datefmt='%m-%d %H:%M')
    ch.setFormatter(formatter)
    
    logger.addHandler(ch)
    if file:    
        fh=logging.FileHandler("E:/wangzhilin/QuantumGalaxy/logs/%s.log"%file,'a',encoding='utf-8')
        fh.setLevel(logging.INFO)
        fh.setFormatter(formatter)
        logger.addHandler(fh)
    return logger


In [3]:
logger=get_logger('logger')

In [4]:
path='E:\wangzhilin\QuantumGalaxy\\backbone\\backbone_catelog'

In [5]:
files=[]

In [6]:
for file in os.listdir(path):
    if file.endswith('.csv'):
        name=file.split('.')[0]
        files.append((name,os.path.join(path,file)))

In [7]:
files

[('光伏', 'E:\\wangzhilin\\QuantumGalaxy\\backbone\\backbone_catelog\\光伏.csv'),
 ('农牧业', 'E:\\wangzhilin\\QuantumGalaxy\\backbone\\backbone_catelog\\农牧业.csv'),
 ('半导体', 'E:\\wangzhilin\\QuantumGalaxy\\backbone\\backbone_catelog\\半导体.csv'),
 ('海运', 'E:\\wangzhilin\\QuantumGalaxy\\backbone\\backbone_catelog\\海运.csv'),
 ('煤炭', 'E:\\wangzhilin\\QuantumGalaxy\\backbone\\backbone_catelog\\煤炭.csv'),
 ('电力', 'E:\\wangzhilin\\QuantumGalaxy\\backbone\\backbone_catelog\\电力.csv'),
 ('石油化工',
  'E:\\wangzhilin\\QuantumGalaxy\\backbone\\backbone_catelog\\石油化工.csv'),
 ('石油天然气',
  'E:\\wangzhilin\\QuantumGalaxy\\backbone\\backbone_catelog\\石油天然气.csv'),
 ('钢铁', 'E:\\wangzhilin\\QuantumGalaxy\\backbone\\backbone_catelog\\钢铁.csv'),
 ('锂离子电池',
  'E:\\wangzhilin\\QuantumGalaxy\\backbone\\backbone_catelog\\锂离子电池.csv')]

In [8]:
backbone=pd.DataFrame(columns=['id','backbone'])
for n,f in files:
    df=pd.read_csv(f)
    df.rename(columns={'~id':'id'},inplace=True)
    df=df[['id','name']]
    df.insert(1,'backbone',n)
    backbone=pd.concat([backbone,df])

In [9]:
backbone

,id,backbone,name
0,254,光伏,光伏组件
1,427,光伏,串焊机
2,428,光伏,硅片分选机
3,429,光伏,光伏硅片
4,430,光伏,光伏电池片
...,...,...,...
76,4711,锂离子电池,容百科技
77,5099,锂离子电池,导电用炭黑
78,5535,锂离子电池,国轩高科
79,5962,锂离子电池,光储一体化


In [10]:
data={}

In [11]:
for id,belong in backbone.groupby('id'):
    data[id]=str(belong.backbone.values.tolist())[1:-1]

In [12]:
data

{6: "'电力'",
 14: "'石油化工'",
 15: "'煤炭', '石油天然气'",
 21: "'煤炭'",
 22: "'煤炭'",
 27: "'海运'",
 41: "'农牧业'",
 54: "'农牧业'",
 61: "'海运'",
 68: "'电力'",
 73: "'石油化工'",
 164: "'海运', '钢铁'",
 170: "'煤炭'",
 178: "'石油化工'",
 181: "'石油化工'",
 199: "'石油化工'",
 207: "'石油化工'",
 213: "'煤炭'",
 241: "'农牧业'",
 254: "'光伏', '石油化工'",
 255: "'电力'",
 328: "'钢铁'",
 382: "'半导体'",
 390: "'石油天然气'",
 393: "'石油化工'",
 417: "'锂离子电池'",
 426: "'煤炭'",
 427: "'光伏'",
 428: "'光伏'",
 429: "'光伏'",
 430: "'光伏'",
 431: "'光伏'",
 432: "'光伏', '电力', '锂离子电池'",
 433: "'光伏'",
 456: "'半导体'",
 457: "'半导体'",
 465: "'煤炭'",
 511: "'锂离子电池'",
 517: "'锂离子电池'",
 518: "'海运'",
 531: "'海运', '石油化工', '石油天然气'",
 532: "'石油化工'",
 533: "'石油天然气'",
 534: "'石油化工', '石油天然气'",
 539: "'锂离子电池'",
 540: "'锂离子电池'",
 542: "'石油化工', '锂离子电池'",
 543: "'石油化工', '锂离子电池'",
 544: "'锂离子电池'",
 547: "'石油化工', '锂离子电池'",
 552: "'锂离子电池'",
 553: "'锂离子电池'",
 554: "'锂离子电池'",
 555: "'锂离子电池'",
 556: "'锂离子电池'",
 560: "'锂离子电池'",
 563: "'锂离子电池'",
 565: "'锂离子电池'",
 566: "'锂离子电池'",
 567: "'锂离子电池'

In [13]:
uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
user = "QG_Editor"
password = "editor"
neo=Neo4j(uri,user,password)

In [14]:
neo.simple_query('match (n) remove n.backbone')


10-19 11:39 logger       INFO     accept cypher query match (n) remove n.backbone
10-19 11:39 logger       INFO     get 0 records


[]

In [15]:

for k,v in data.items():
    cypher='match (n) where id(n)=%d set n.backbone="%s"'%(k,v)

    print(cypher)
    neo.simple_query(cypher)

10-19 11:39 logger       INFO     accept cypher query match (n) where id(n)=6 set n.backbone="'电力'"


match (n) where id(n)=6 set n.backbone="'电力'"


10-19 11:39 logger       INFO     get 0 records
10-19 11:39 logger       INFO     accept cypher query match (n) where id(n)=14 set n.backbone="'石油化工'"


match (n) where id(n)=14 set n.backbone="'石油化工'"


10-19 11:39 logger       INFO     get 0 records
10-19 11:39 logger       INFO     accept cypher query match (n) where id(n)=15 set n.backbone="'煤炭', '石油天然气'"
10-19 11:39 logger       INFO     get 0 records
10-19 11:39 logger       INFO     accept cypher query match (n) where id(n)=21 set n.backbone="'煤炭'"


...

match (n) where id(n)=6061 set n.backbone="'海运'"


10-19 11:40 logger       INFO     accept cypher query match (n) where id(n)=6062 set n.backbone="'海运'"
10-19 11:40 logger       INFO     get 0 records


match (n) where id(n)=6062 set n.backbone="'海运'"


10-19 11:40 logger       INFO     accept cypher query match (n) where id(n)=6063 set n.backbone="'海运'"
10-19 11:40 logger       INFO     get 0 records


match (n) where id(n)=6063 set n.backbone="'海运'"
